In [1]:
import subprocess
import os

def compile_to_ir(testcase, llvm_path, level):
    compiler = os.path.join(llvm_path, testcase["compiler"])
    input_file = testcase["input_file"]

    cmd = [compiler, "-S", "-emit-llvm", *testcase["flags"], level, 
           "-Xclang", "-disable-llvm-passes", "-o", "-", input_file]
    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        # print(f"[OK] {testcase['id']} ({len(result.stdout)} chars)")
        return result.stdout
    except subprocess.CalledProcessError as e:
        # print(f"[FAIL] {testcase['id']}\nstderr: {e.stderr}")
        raise Exception(f"[FAIL] {testcase['id']} stderr: {e.stderr}")

testcase = {
    "compiler": "clang",
    "flags" : [],
    "input_file": "/home/dxic/llvmtest/test/twofunc.c",
    "id": "twofunc"
}
llvm_path = "/home/dxic/llvmlearn/llvm-project/llvm/build/bin/"    
if __name__ == "__main__":
    ir = compile_to_ir(testcase, llvm_path, "-O3")
    print(ir)

In [2]:
import re

def run_opt_pipeline(ir, opt_pipeline_path, level):
    """
    运行 build/opt-pipeline，解析 stderr 中的 diff 统计。

    Returns:
        list[dict]: 每个 pass 一条记录
            {pass, ir, funcs: [{name, L_orig, L_new, BB_orig, BB_add, BB_del, BB_mod,
                                  Inst_orig, Inst_add, Inst_del, Inst_mod,
                                  NBB_add, NBB_del, NBB_mod,
                                  NI_add, NI_del, NI_mod, destroyed}]}
    """
    result = subprocess.run([opt_pipeline_path, level], input=ir,
                            capture_output=True, text=True)

    return result.stderr


opt_path = "/home/dxic/LoopOptDiff2/RQ1/build/opt-pipeline"
if __name__ == "__main__":

    if ir:
        stderr = run_opt_pipeline(ir, opt_path, "-O3")
        print(stderr)

In [3]:
import re


def parse_opt_stderr(stderr):
    """从 opt-pipeline 的 stderr 输出中解析出结构化统计。

    Returns:
        list[dict]: 每个 pass 一条
    """
    func_re = re.compile(
        r'^\s+(\S+)\s+L\((\d+)->(\d+)\)\s+BB\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'Inst\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'NBB\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'NI\((\d+)\s+\+(\d+)\s+-(\d+)\s+~(\d+)\)\s+'
        r'(?:NEBB\(\+(\d+),-(\d+)\)\s*)?'
        r'(?:\s+\[DESTROYED\])?'
    )
    records = []
    cur = None
    for line in stderr.splitlines():
        if line.startswith('['):
            parts = line[1:].split(']', 1)
            rest = parts[1].strip() if len(parts) > 1 else ""
            invalidated = rest.endswith(' [INVALIDATED]')
            if invalidated:
                rest = rest[:-len(' [INVALIDATED]')]
            cur = {"pass": parts[0].strip(), "ir": rest, "invalidated": invalidated, "funcs": []}
            records.append(cur)
        elif cur:
            m = func_re.match(line)
            if m:
                g = m.groups()
                cur["funcs"].append({
                    "name": g[0],
                    "L_orig": int(g[1]), "L_new": int(g[2]),
                    "BB_orig": int(g[3]), "BB_add": int(g[4]),
                    "BB_del": int(g[5]), "BB_mod": int(g[6]),
                    "Inst_orig": int(g[7]), "Inst_add": int(g[8]),
                    "Inst_del": int(g[9]), "Inst_mod": int(g[10]),
                    "NBB_orig": int(g[11]), "NBB_add": int(g[12]),
                    "NBB_del": int(g[13]), "NBB_mod": int(g[14]),
                    "NI_orig": int(g[15]), "NI_add": int(g[16]),
                    "NI_del": int(g[17]), "NI_mod": int(g[18]),
                    "NEBB_add": int(g[19]) if g[19] is not None else 0,
                    "NEBB_del": int(g[20]) if g[20] is not None else 0,
                    "destroyed": "[DESTROYED]" in line,
                })
    return records


if __name__ == "__main__":
    records = parse_opt_stderr(stderr)
    print(f"Total passes: {len(records)}")
    for r in records:
        print(r)
        # for f in r["funcs"]:
            # if f["BB_add"] or f["BB_del"] or f["BB_mod"] or f["Inst_add"] or f["Inst_del"] or f["Inst_mod"]:
            #     print(f"  [{r['pass']}] {f['name']}: BB(+{f['BB_add']} -{f['BB_del']} ~{f['BB_mod']}) Inst(+{f['Inst_add']} -{f['Inst_del']} ~{f['Inst_mod']})")
            #     break

In [4]:
# loopoptpass=["LoopSimplifyPass", "LCSSAPass", "LoopInstSimplifyPass", "LoopSimplifyCFGPass", 
#              "LICMPass", "LoopRotatePass", "SimpleLoopUnswitchPass", 
#              "LoopIdiomRecognizePass", "IndVarSimplifyPass", "LoopDeletionPass", "LoopFullUnrollPass",
#              "LoopVectorizePass", "LoopLoadEliminationPass", "LoopUnrollPass",
#              "LoopStrengthReducePass"]
loopoptpass = {
    "LoopSimplifyPass":"LoopSimplify", 
    "LCSSAPass":"LCSSA",
    "LoopInstSimplifyPass":"LoopInstSimplify",
    "LoopSimplifyCFGPass":"LoopSimplifyCFG", 
    "LICMPass":"LICM", 
    "LoopRotatePass":"LoopRotate",
    "SimpleLoopUnswitchPass":"SimpleLoopUnswitch", 
    "LoopIdiomRecognizePass":"LoopIdiomRecognize", 
    "IndVarSimplifyPass":"IndVarSimplify", 
    "LoopDeletionPass":"LoopDeletion", 
    "LoopFullUnrollPass":"LoopFullUnroll",
    "LoopVectorizePass":"LoopVectorize",
    "LoopLoadEliminationPass":"LoopLoadElimination", 
    "LoopUnrollPass":"LoopUnroll",
    "LoopStrengthReducePass":"LoopStrengthReduce"
}

In [5]:
def summary(records, loopoptpass):
    """统计每个函数被所有 pass 和 Loop pass 操作的 BB、Inst、NBB、NI 数量。

    Returns:
        dict: {func_name: {total: {BB_add, BB_del, Inst_add, Inst_del,
                                   NBB_add, NBB_del, NI_add, NI_del},
                           loopopt: {...},
                           pass_name: {...}}}
    """
    from collections import defaultdict
    blank = lambda: {"BB_add": 0, "BB_del": 0, "Inst_add": 0, "Inst_del": 0,
                     "NBB_add": 0, "NBB_del": 0, "NI_add": 0, "NI_del": 0, 
                     "NEBB_add":0, "NEBB_del":0}
    keys = list(blank().keys())
    result = defaultdict(lambda: {"total": blank(), "loopopt": blank(),
                                  "passes": defaultdict(blank),
                                  "L_init": None, "L_max": 0})
    for r in records:
        is_loop_opt = r["pass"] in loopoptpass
        for f in r["funcs"]:
            d = result[f["name"]]
            if d["L_init"] is None:
                d["L_init"] = f["L_orig"]
            if d["L_max"] < f["L_orig"]:
                d["L_max"] = f["L_orig"]
            for k in keys:
                d["total"][k] += f[k]
                if is_loop_opt:
                    d["loopopt"][k] += f[k]
                    d["passes"][r["pass"]][k] += f[k]
    return {fn: {"total": v["total"],
                 "loopopt": v["loopopt"],
                 "L_init": v["L_init"],
                 "L_max": v["L_max"],
                 **v["passes"]}
            for fn, v in result.items()}

import sys
def print_summary(sum, file=sys.stdout):
    for fn, stats in sum.items():
        t = stats["total"]
        l = stats["loopopt"]
        print(f"{fn:20s} {"total":30s} BB(+{t['BB_add']} -{t['BB_del']}) Inst(+{t['Inst_add']} -{t['Inst_del']})"
              f" NBB(+{t['NBB_add']} -{t['NBB_del']}) NI(+{t['NI_add']} -{t['NI_del']})"
              f" NEBB(+{t['NEBB_add']},-{t['NEBB_del']})", file=file)
        print(f"{str((stats['L_init'], stats['L_max'])):20s} {"loopopt":30s} BB(+{l['BB_add']} -{l['BB_del']}) Inst(+{l['Inst_add']} -{l['Inst_del']})"
              f" NBB(+{l['NBB_add']} -{l['NBB_del']}) NI(+{l['NI_add']} -{l['NI_del']})"
              f" NEBB(+{l['NEBB_add']},-{l['NEBB_del']})", file=file)
        for pn, ps in stats.items():
            if pn in ("total", "loopopt", "L_init", "L_max"):
                continue
            if ps["BB_add"] or ps["BB_del"] or ps["Inst_add"] or ps["Inst_del"]:
                print(f"{'':20s} {pn:30s} BB(+{ps['BB_add']} -{ps['BB_del']})"
                      f" Inst(+{ps['Inst_add']} -{ps['Inst_del']})"
                      f" NBB(+{ps['NBB_add']} -{ps['NBB_del']})"
                      f" NI(+{ps['NI_add']} -{ps['NI_del']})"
                      f" NEBB(+{ps['NEBB_add']},-{ps['NEBB_del']})", file=file)


# 测试
if __name__ == "__main__":
    s = summary(records, loopoptpass)
    print_summary(s)

In [6]:
import sys


def single_count(testcase, level, stderr_file=None):
    ir = compile_to_ir(testcase, llvm_path, level)
    stderr = run_opt_pipeline(ir, opt_path, level)
    records = parse_opt_stderr(stderr)
    sum = summary(records, loopoptpass)
    if stderr_file:
        with open(stderr_file, "w") as f:
            f.write(stderr)
            f.write("========summary========\n")
            print_summary(sum, file=f)
    return sum

def single_count_from_stderr(testcase, level, stderr_file):
    with open(stderr_file) as f:
        stderr = f.read()
    stderr = stderr.split("========summary========")[0]
    records = parse_opt_stderr(stderr)
    return summary(records, loopoptpass)

if __name__ == "__main__":
    sum  = single_count(testcase, "-O3", 'test/twofuncO3.txt')
    sum2  = single_count_from_stderr(testcase, "-O3", 'test/twofuncO3.txt')
    print(sum==sum2)

In [7]:
from pathlib import Path

error_list = []

def batch_count(testcases, level, output_dir, workers, single_count_func):
    """统计多个 testcase 的 NBB/NI 占比和 per-loop-pass 计数。

    Returns:
        dict: {nbb_ratio: [...], ni_ratio: [...],
               loop_passes: {pass_name: {"NBB_add": ..., "NBB_del": ..., "NI_add": ..., "NI_del": ...}}}
    """
    from collections import defaultdict
    from concurrent.futures import ThreadPoolExecutor, as_completed
    
    result = {"nbb_ratio": {}, "ni_ratio": {}, "bb_ratio": {}, "inst_ratio": {}, "nebb_ratio": {}, "loop_passes": {}}
    keys = ["NBB_add", "NBB_del", "NI_add", "NI_del", "BB_add", "BB_del", "Inst_add", "Inst_del", "NEBB_add", "NEBB_del"]
    blank = lambda: {k:0 for k in keys}
    pass_sum = defaultdict(blank)

    global error_list
    with ThreadPoolExecutor(max_workers=workers) as executor:
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            futures = {executor.submit(single_count_func, tc, level, Path(output_dir)/f"{tc['id']}.txt"): tc for tc in testcases}
        else:
            futures = {executor.submit(single_count_func, tc, level): tc for tc in testcases}
        
        for fut in as_completed(futures):
            tc = futures[fut]
            try:
                s = fut.result()
            except Exception as e:
                print(f"Error processing {tc['id']}: {e}")
                error_list.append((tc['id'],e))
                continue
            for fn, stats in s.items():
                if stats["L_init"] is None or stats["L_init"] <= 0:
                    continue
                # if stats["L_max"] <= 0:
                #     continue
                key = f"{tc['id']}::{fn}"

                t_nbb = stats["total"]["NBB_add"] + stats["total"]["NBB_del"]
                l_nbb = stats["loopopt"]["NBB_add"] + stats["loopopt"]["NBB_del"]
                nbb_ratio = l_nbb / t_nbb if t_nbb else 0
                result["nbb_ratio"][key] = nbb_ratio

                t_ni = stats["total"]["NI_add"] + stats["total"]["NI_del"]
                l_ni = stats["loopopt"]["NI_add"] + stats["loopopt"]["NI_del"]
                ni_ratio = l_ni / t_ni if t_ni else 0
                result["ni_ratio"][key] = ni_ratio
                
                t_bb = stats["total"]["BB_add"] + stats["total"]["BB_del"]
                l_bb = stats["loopopt"]["BB_add"] + stats["loopopt"]["BB_del"]
                bb_ratio = l_bb / t_bb if t_bb else 0
                result["bb_ratio"][key] = bb_ratio
                
                t_i = stats["total"]["Inst_add"] + stats["total"]["Inst_del"]
                l_i = stats["loopopt"]["Inst_add"] + stats["loopopt"]["Inst_del"]
                i_ratio = l_i / t_i if t_i else 0
                result["inst_ratio"][key] = i_ratio
                
                t_nebb = stats["total"]["NEBB_add"] + stats["total"]["NEBB_del"]
                l_nebb = stats["loopopt"]["NEBB_add"] + stats["loopopt"]["NEBB_del"]
                nebb_ratio = l_nebb / t_nebb if t_nebb else 0
                result["nebb_ratio"][key] = nebb_ratio
                
                print(f"{key}: NBB_ratio={nbb_ratio:.2%} NI_ratio={ni_ratio:.2%}, BB_ratio={bb_ratio:.2%}, Inst_ratio={i_ratio:.2%}, NEBB_ratio={nebb_ratio:.2%}")

                for pn, ps in stats.items():
                    if pn in ("total", "loopopt", "L_init", "L_max"):
                        continue
                    for k in keys:
                        pass_sum[pn][k] += ps[k]

    result["loop_passes"] = pass_sum

    return result


# 测试
if __name__ == "__main__":
    import json
    with open("test/testsuitex.json", "r") as f:
        testsuite = json.load(f)
    r = batch_count(testsuite[:3], "-O3", "./test/O3", 8, single_count)

    print("="*30)
    print(f"NBB_ratio={r['nbb_ratio']}")
    print(f"NI_ratio={r['ni_ratio']}")
    print("\nLoop pass NBB/NI totals:")
    for pn, ps in r["loop_passes"].items():
        if ps["BB_add"] or ps["BB_del"] or ps["Inst_add"] or ps["Inst_del"]:
                print(f"{'':9s} {pn:30s} BB(+{ps['BB_add']} -{ps['BB_del']})"
                      f" Inst(+{ps['Inst_add']} -{ps['Inst_del']})"
                      f" NBB(+{ps['NBB_add']} -{ps['NBB_del']})"
                      f" NI(+{ps['NI_add']} -{ps['NI_del']})"
                      f" NEBB(+{ps['NEBB_add']},-{ps['NEBB_del']})")
    del testsuite

In [ ]:
if __name__ == "__main__":
    import json
    with open("/home/dxic/LoopOptDiff2/extract/testsuite260808.json", "r") as f:
        testsuite = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/coreutils260808.json", "r") as f:
        coreutils = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/openssl260808.json", "r") as f:
        openssl = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/ffmpeg260808.json", "r") as f:
        ffmpeg = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/curl260808.json", "r") as f:
        curl = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/zlib260808.json", "r") as f:
        zlib = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/sqlite260808.json", "r") as f:
        sqlite = json.load(f)
    with open("/home/dxic/LoopOptDiff2/extract/csmith260808.json", "r") as f:
        csmith = json.load(f)
    from pathlib import Path
    result_path = Path("test/260811")
    tcs =  testsuite+coreutils+openssl+ffmpeg
    error_list = []
    rO3 = batch_count(tcs, "-O3", result_path/"O3", 16, single_count)
    rO2 = batch_count(tcs, "-O2", result_path/"O2", 16, single_count)
    rOs = batch_count(tcs, "-Os", result_path/"Os", 16, single_count)
    rO1 = batch_count(tcs, "-O1", result_path/"O1", 16, single_count)


    # rO3 = batch_count(tcs, "-O3", result_path/"O3", 16, single_count_from_stderr)
    # rO2 = batch_count(tcs, "-O2", result_path/"O2", 16, single_count_from_stderr)
    # rOs = batch_count(tcs, "-Os", result_path/"Os", 16, single_count_from_stderr)
    # rO1 = batch_count(tcs, "-O1", result_path/"O1", 16, single_count_from_stderr)

    print(error_list)


In [9]:
if __name__ == "__main__":
    print(len(rO3['nbb_ratio']))
    print(len(rO2['nbb_ratio']))
    print(len(rOs['nbb_ratio']))
    print(len(rO1['nbb_ratio']))
    print(set(rO3['nbb_ratio'])-set(rOs['nbb_ratio']))
    print(set(rOs['nbb_ratio'])-set(rO3['nbb_ratio']))
